# Лабораторная работа 2
## Вариант на 3
### Работа с `Ollama` и `Qwen2.5:0.5b`

В ходе лабораторной работы нужно поднять локальный сервер `Ollama`, подключить к нему модель `Qwen2.5:0.5b`, отправить 10 запросов и сохранить ответы в отдельный отчёт.

Цель работы: настроить локальный запуск LLM через `Ollama`, проверить отправку запросов по HTTP и получить итоговую таблицу с запросами и ответами модели.

## Что нужно сделать перед запуском

Перед выполнением ячеек нужно один раз подготовить окружение.

1. Установить `Ollama` с официального сайта: [https://ollama.com/download](https://ollama.com/download)
2. Скачать модель командой:

```powershell
ollama pull qwen2.5:0.5b
```

3. Запустить сервер `Ollama` в отдельном терминале:

```powershell
ollama serve
```

4. Установить Python-библиотеки, если они ещё не стоят:

```powershell
pip install requests pandas
```

## Подключение библиотек и основные настройки

В этой части подключаются библиотеки и задаются основные параметры: адрес локального сервера, название модели и путь, по которому будет сохранён итоговый отчёт.

Отчёт будем сохранять в файл `inference_report.csv` в текущей папке проекта.

In [8]:
from pathlib import Path
import json
from typing import Any, Dict, List

import pandas as pd
import requests

project_root = Path.cwd()
report_path = project_root / "inference_report.csv"

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
MODEL_NAME = "qwen2.5:0.5b"

print("Текущая папка проекта:", project_root)
print("Путь к отчёту:", report_path)
print("Адрес Ollama:", OLLAMA_GENERATE_URL)
print("Модель:", MODEL_NAME)

Текущая папка проекта: m:\Учёба\8-й сем\ИИ
Путь к отчёту: m:\Учёба\8-й сем\ИИ\inference_report.csv
Адрес Ollama: http://localhost:11434/api/generate
Модель: qwen2.5:0.5b


## Проверка доступности сервера

Сначала полезно проверить, отвечает ли `Ollama` по HTTP. Это позволит сразу понять, можно ли отправлять запросы к модели.

Если всё запущено правильно, эта проверка вернёт `True`.

In [9]:
def check_ollama_server(base_url: str = OLLAMA_BASE_URL, timeout: int = 10) -> bool:
    """Проверяет, доступен ли локальный сервер Ollama по HTTP."""
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=timeout)
        response.raise_for_status()
        return True
    except requests.RequestException as exc:
        print("Не удалось подключиться к Ollama:", exc)
        return False


server_ok = check_ollama_server()
print("Сервер доступен:", server_ok)

Сервер доступен: True


## Проверка загруженных моделей

Нам нужна модель `Qwen2.5:0.5b`, поэтому ниже выводится список локально установленных моделей. Так можно сразу проверить, скачалась ли нужная модель и правильно ли работает сервер.

In [10]:
def list_local_models(base_url: str = OLLAMA_BASE_URL, timeout: int = 10) -> Dict[str, Any]:
    """Возвращает список локально доступных моделей Ollama в формате JSON."""
    response = requests.get(f"{base_url}/api/tags", timeout=timeout)
    response.raise_for_status()
    return response.json()


if server_ok:
    models_payload = list_local_models()
    print(json.dumps(models_payload, ensure_ascii=False, indent=2))
else:
    print("Сервер недоступен, список моделей получить нельзя.")

{
  "models": [
    {
      "name": "qwen2.5:0.5b",
      "model": "qwen2.5:0.5b",
      "modified_at": "2026-04-23T13:46:36.7994598+03:00",
      "size": 397821319,
      "digest": "a8b0c51577010a279d933d14c2a8ab4b268079d44c5c8830c0a93900f1827c67",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "qwen2",
        "families": [
          "qwen2"
        ],
        "parameter_size": "494.03M",
        "quantization_level": "Q4_K_M"
      }
    }
  ]
}


## Один пробный запрос

Перед тем как запускать сразу 10 запросов, удобно сделать одну простую проверку. Если модель вернула нормальный текстовый ответ, значит всё подключено правильно и можно переходить к основной части.

In [12]:
def generate_text(
    prompt: str,
    model: str = MODEL_NAME,
    url: str = OLLAMA_GENERATE_URL,
    timeout: int = 120,
) -> str:
    """Отправляет один запрос в Ollama HTTP API и возвращает текст ответа модели."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }

    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    return data.get("response", "").strip()


test_prompt = "Привет, как жизнь? И ответь, пожалуйста, на русском"

if server_ok:
    test_response = generate_text(test_prompt)
    print("Запрос:")
    print(test_prompt)
    print("\nОтвет модели:")
    print(test_response)
else:
    print("Сервер недоступен, пробный запрос пропущен.")

Запрос:
Привет, как жизнь? И ответь, пожалуйста, на русском

Ответ модели:
Привет! Жизнь – это не только простой набор событий и действий, но и сложная система состояния и обстоятельств. Она может быть полезной для людей в различных жизненных ситуациях:

1. Общение: Постоянный контакт с людьми и информацией о окружающем мире.

2. Работа: Взаимодействие с различными профессионалами, связанными с искусственным интеллектом или другими технологиями.

3. Семья и детство: Сохранение отношений с родственниками и детей.

4. Здоровье: Проведение естественных физических и эмоциональных требований.

5. Образование: Создание и распространение информации для взрослых и детей.

6. Финансы: Сбор средств, инвестиции в бизнес или другие активы.

7. Экология: Попытки построения благоустроенного мира вокруг нас.

8. Спорт: Регulation и развитие техник и техноэмоциональных процессов.

9. Обучение: Создание и обучение новых навыков для дальнейшего развития.

10. Практика: Опыт в различных областях, таких к

## Набор из 10 запросов

По заданию нужно прогнать скрипт через 10 произвольных запросов и зафиксировать ответы модели. Ниже подготовлен список из десяти запросов на разные темы, чтобы было видно, как LLM отвечает на обычные пользовательские сообщения.

In [13]:
prompts: List[str] = [
    "Какую клавиатуру посоветуешь, механическую или мембранную?",
    "Какая азиатская страна в 19 веке развивалась стремительнее всех?",
    "Чем отличается Python от C++ в двух-трёх предложениях?",
    "Придумай короткий план подготовки к экзамену на 3 дня.",
    "Посоветуй фильм на вечер.",
    "Левая или правая палочка Твикс?",
    "Расскажи в кратце отличие ВПН от прокси.",
    "Какая сейчас погода в Белграде?",
    "Как отмыть разводы от чая на кружке?",
    "Посчитай от нуля до нуля"
]

print(f"Количество запросов: {len(prompts)}")
for index, prompt in enumerate(prompts, start=1):
    print(f"{index}. {prompt}")

Количество запросов: 10
1. Какую клавиатуру посоветуешь, механическую или мембранную?
2. Какая азиатская страна в 19 веке развивалась стремительнее всех?
3. Чем отличается Python от C++ в двух-трёх предложениях?
4. Придумай короткий план подготовки к экзамену на 3 дня.
5. Посоветуй фильм на вечер.
6. Левая или правая палочка Твикс?
7. Расскажи в кратце отличие ВПН от прокси.
8. Какая сейчас погода в Белграде?
9. Как отмыть разводы от чая на кружке?
10. Посчитай от нуля до нуля


## Прогон из 10 запросов

Следующая ячейка по очереди отправляет все 10 запросов на сервер `Ollama`, получает ответы модели и собирает их в таблицу `pandas`.

В консоль выводится короткая информация по каждому шагу, чтобы было видно, что скрипт действительно работает, а не завис.

In [14]:
def run_inference_batch(prompts_list: List[str], model: str = MODEL_NAME) -> pd.DataFrame:
    """Прогоняет список запросов через модель и возвращает таблицу с результатами."""
    rows = []

    for index, prompt in enumerate(prompts_list, start=1):
        print(f"[{index}/{len(prompts_list)}] Отправляется запрос")
        response_text = generate_text(prompt=prompt, model=model)

        print("Запрос:", prompt)
        print("Ответ модели:", response_text[:250] + ("..." if len(response_text) > 250 else ""))
        print("-" * 80)

        rows.append(
            {
                "prompt_id": index,
                "prompt": prompt,
                "response": response_text,
            }
        )

    return pd.DataFrame(rows)


if server_ok:
    results_df = run_inference_batch(prompts)
    results_df
else:
    print("Сервер недоступен, массовый прогон пропущен.")

[1/10] Отправляется запрос
Запрос: Какую клавиатуру посоветуешь, механическую или мембранную?
Ответ модели: Конечно! Важно выбирать наиболее подходящую для конкретного задачи и технических возможностей. Вот некоторые преимущества выбора механической клавиатуры:

1. Техническая поддержка: Механическое управление имеет больший опыт в системах контроля и упра...
--------------------------------------------------------------------------------
[2/10] Отправляется запрос
Запрос: Какая азиатская страна в 19 веке развивалась стремительнее всех?
Ответ модели: В 19 веке, азиатские страны, как правило, не имели такой революционной и гибридной формы развития как США или Великобритания. Однако, существовали и другие важные азиатские страны, которые развивались быстрее:

1. Швейцария: Азия, в основном, отстаив...
--------------------------------------------------------------------------------
[3/10] Отправляется запрос
Запрос: Чем отличается Python от C++ в двух-трёх предложениях?
Ответ модели: Pyth

## Сохранение отчёта

In [16]:
def prepare_report_table(df: pd.DataFrame) -> pd.DataFrame:
    """Подготавливает таблицу результатов в более читаемом виде для сохранения."""
    export_df = df[["prompt_id", "prompt", "response"]].copy()
    export_df = export_df.rename(
        columns={
            "prompt_id": "Номер запроса",
            "prompt": "Запрос",
            "response": "Ответ модели",
        }
    )

    for column in ["Запрос", "Ответ модели"]:
        export_df[column] = (
            export_df[column]
            .str.replace(r"\s*\n+\s*", " | ", regex=True)
            .str.replace(r"\s{2,}", " ", regex=True)
            .str.strip()
        )

    return export_df


def save_inference_report(df: pd.DataFrame, output_path: Path = report_path) -> None:
    """Сохраняет таблицу с результатами инференса в CSV-файл."""
    export_df = prepare_report_table(df)
    export_df.to_csv(output_path, index=False, encoding="utf-8-sig", sep=";")


if server_ok:
    save_inference_report(results_df)
    print("Отчёт сохранён в:", report_path)
else:
    print("Сервер недоступен, отчёт не сохранён.")

Отчёт сохранён в: m:\Учёба\8-й сем\ИИ\inference_report.csv


## Просмотр итоговой таблицы

Выводим результаты в таблицу

In [18]:
if server_ok:
    final_report_df = prepare_report_table(results_df)
    final_report_df
else:
    print("Нет данных для отображения.")

## Короткий вывод

В ходе выполнения лабораторной работы был успешно использован локальный сервер `Ollama` с моделью `Qwen2.5:0.5b`. С помощью Python и библиотеки `requests` были отправлены запросы на HTTP-адрес сервера, после чего модель вернула текстовые ответы.

Далее был выполнен прогон 10 произвольных запросов, а результаты сохранены в файл `inference_report.csv`. В итоговом отчёте для каждого запроса указаны его номер, текст самого запроса и ответ модели.

По результатам можно сделать вывод, что система работает корректно: сервер принимает запросы, модель отвечает, а данные успешно сохраняются в отчёт. При этом качество ответов получилось неоднозначным: на части запросов модель ответила нормально, а на части дала неточные или странные формулировки.